# External stress timing breakdown

Profiles **external** stress (Output 3-1 feeders): load → per-scenario wall time →
pipeline stages for a heavy scenario (B3) → `external_interest()` call cost.

Use this to see where the 3–5 minute suite spend goes. Not an SUT.

In [1]:
from __future__ import annotations

import time
from contextlib import contextmanager
from pathlib import Path

import pandas as pd

from lic_dsf.load import (
    load_core,
    load_input6_standard,
    load_input7_residual_params,
    load_tailored_params,
)
from lic_dsf.load.tailored import load_customized_spec
from lic_dsf.books.macro.book import MacroDebtBook
from lic_dsf.stress.shocks.bound import external_residual_borrowing
from lic_dsf.stress.context import StressContext
from lic_dsf.stress.external_dynamics import ExternalDebtDynamics
from lic_dsf.resfin import ResidualFinancingEngine
from lic_dsf.stress.ratios.external import StressExternalRatios
from lic_dsf.stress.runner.coupled import CoupledScenarioRunner
from lic_dsf.stress.runner.external import ExternalScenarioRunner
from lic_dsf.stress.shocks import MacroShockFactory
from lic_dsf.stress.spec import ScenarioRegistry, ShockKind
from lic_dsf.stress.suite import StressSuite
from lic_dsf.stress.shocks.tailored import applicable_tailored_ids

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


@contextmanager
def timed(label: str, rows: list[dict]):
    t0 = time.perf_counter()
    yield
    rows.append({"section": label, "seconds": time.perf_counter() - t0})


def timing_frame(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    total = float(frame["seconds"].sum())
    frame["share_%"] = 100.0 * frame["seconds"] / total if total else 0.0
    return frame


WORKBOOK

PosixPath('/home/sravan/py-lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

## 1. Load

In [2]:
load_rows: list[dict] = []

with timed("load_core", load_rows):
    macro, external, ext_base, pub_base = load_core(WORKBOOK)
with timed("load_input6_standard", load_rows):
    input6 = load_input6_standard(WORKBOOK)
with timed("load_input7_residual_params", load_rows):
    residual = load_input7_residual_params(WORKBOOK)
with timed("load_tailored_params + custom_spec", load_rows):
    tailored = load_tailored_params(WORKBOOK)
    custom_spec = load_customized_spec(WORKBOOK)
with timed("StressContext.from_parts", load_rows):
    ctx = StressContext.from_parts(
        macro,
        external,
        input6,
        residual,
        ext_base=ext_base,
        pub_base=pub_base,
        tailored=tailored,
        custom_spec=custom_spec,
    )

load_timing = timing_frame(load_rows)
load_timing

,section,seconds,share_%
0,load_core,29.776,98.157
1,load_input6_standard,0.148,0.486
2,load_input7_residual_params,0.120,0.396
3,load_tailored_params + custom_spec,0.291,0.960
4,StressContext.from_parts,0.000,0.000


## 2. Per-scenario wall time (standard external + B2)

Same set as `run_standard_external_stress` (A1 included here so you can see its cost;
the facade often drops A1 from the returned dict).

In [3]:
ext_runner = ExternalScenarioRunner(context=ctx)
coupled_runner = CoupledScenarioRunner(context=ctx)

scenario_rows: list[dict] = []
results = {}

standard_ids = [
    sid
    for sid, spec in ScenarioRegistry.STANDARD.items()
    if spec.output_binding.output_31_source == "external"
] + ["B2_PrimaryBalance"]

for sid in standard_ids:
    spec = ScenarioRegistry.get(sid)
    t0 = time.perf_counter()
    if sid == "B2_PrimaryBalance":
        result = coupled_runner.run(spec)
    else:
        result = ext_runner.run(spec)
    elapsed = time.perf_counter() - t0
    results[sid] = result
    gap_iters = result.external_gap.iterations if result.external_gap else None
    scenario_rows.append(
        {
            "scenario": sid,
            "seconds": elapsed,
            "gap_iterations": gap_iters,
            "resfin_iterations": result.resfin.iterations if result.resfin else None,
            "runner": "coupled" if sid == "B2_PrimaryBalance" else "external",
        }
    )

scenario_timing = timing_frame(scenario_rows)
scenario_timing

,scenario,seconds,gap_iterations,resfin_iterations,runner,share_%
0,A1_Historical,18.536,15,15,external,19.721
1,B1_GDP,1.359,0,21,external,1.446
2,B3_Exports,17.909,15,15,external,19.054
3,B4_OtherFlows,17.885,15,15,external,19.028
4,B5_FX,17.718,15,15,external,18.850
5,B6_Combo,17.997,15,15,external,19.147
6,B2_PrimaryBalance,2.589,1,21,coupled,2.754


## 3. Tailored external (A2 / C*)

In [4]:
tailored_rows: list[dict] = []
tailored_ctx = ctx
tailored_runner = ExternalScenarioRunner(context=tailored_ctx)

for sid in applicable_tailored_ids(tailored):
    spec = ScenarioRegistry.get(sid)
    t0 = time.perf_counter()
    result = tailored_runner.run(spec)
    elapsed = time.perf_counter() - t0
    results[sid] = result
    tailored_rows.append(
        {
            "scenario": sid,
            "seconds": elapsed,
            "gap_iterations": (
                result.external_gap.iterations if result.external_gap else None
            ),
            "resfin_iterations": result.resfin.iterations if result.resfin else None,
        }
    )

tailored_timing = timing_frame(tailored_rows)
tailored_timing

,scenario,seconds,gap_iterations,resfin_iterations,share_%
0,A2_Custom,1.289,1,1,5.651
1,C1_CombinedCL,2.655,1,21,11.640
2,C3_Commodity,18.847,15,15,82.612
3,C4_Market,0.022,0,0,0.098


## 4. Pipeline stages for B3 (heavy external path)

Break one scenario into: shock → gap convergence → final ResFin overlay → ratios.

In [5]:
STAGE_SCENARIO = "B3_Exports"
spec = ScenarioRegistry.get(STAGE_SCENARIO)
stage_rows: list[dict] = []

with timed("1_shock_apply", stage_rows):
    shock = MacroShockFactory.from_spec(spec)
    path = shock.apply(ctx, spec)

with timed("2_ExternalDebtDynamics.from_context", stage_rows):
    dynamics = ExternalDebtDynamics.from_context(ctx, path, spec)

with timed("3_compute_gap_converged", stage_rows):
    gap = dynamics.compute_gap_converged()

with timed("4_build_external_overlay_final", stage_rows):
    engine = ResidualFinancingEngine.for_external(
        ctx.residual, path.years, external=ctx.external
    )
    overlay = engine.build_external_overlay(gap.gap)

with timed("5_StressExternalRatios.from_path", stage_rows):
    _ratios = StressExternalRatios.from_path(path, ctx.external, overlay)

stage_timing = timing_frame(stage_rows)
print(
    f"{STAGE_SCENARIO}: gap_iterations={gap.iterations} "
    f"ext_r86_zero={spec.ext_r86_zero}"
)
stage_timing

B3_Exports: gap_iterations=15 ext_r86_zero=False


,section,seconds,share_%
0,1_shock_apply,0.007,0.036
1,2_ExternalDebtDynamics.from_context,0.000,0.000
2,3_compute_gap_converged,19.069,99.954
3,4_build_external_overlay_final,0.002,0.010
4,5_StressExternalRatios.from_path,0.000,0.000


## 5. Inside `compute_gap_converged` (B3)

Time each ResFin iteration: `external_residual_borrowing` vs `build_external_overlay`.

In [6]:
from lic_dsf.stress.external_dynamics import EXTERNAL_INTEREST_TOL, _align, _zero
import lic_dsf.stress.shocks.bound as _bound

iter_rows: list[dict] = []
years = path.years
dyn = ExternalDebtDynamics.from_context(ctx, path, spec)
eng = dyn._resfin_engine()
resfin_interest = _zero(years)
max_iter = 25

for i in range(max_iter):
    t_gap = time.perf_counter()
    gap_i = _bound.external_residual_borrowing(
        dyn.path.baseline,
        dyn.path.shocked,
        resfin_interest=resfin_interest,
        **dyn._borrow_kwargs(),  # type: ignore[arg-type]
    )
    gap_s = time.perf_counter() - t_gap

    if float(gap_i.fillna(0.0).abs().sum()) == 0.0:
        iter_rows.append(
            {
                "iter": i + 1,
                "gap_s": gap_s,
                "overlay_s": 0.0,
                "stopped": "zero_gap",
            }
        )
        break

    t_ov = time.perf_counter()
    overlay_i = eng.build_external_overlay(gap_i)
    ov_s = time.perf_counter() - t_ov
    new_interest = _align(overlay_i.interest, years).fillna(0.0)
    delta = float((new_interest - resfin_interest).abs().max())
    converged = delta < EXTERNAL_INTEREST_TOL
    iter_rows.append(
        {
            "iter": i + 1,
            "gap_s": gap_s,
            "overlay_s": ov_s,
            "interest_delta_max": delta,
            "stopped": "converged" if converged else "",
        }
    )
    if converged:
        break
    resfin_interest = new_interest

iter_timing = pd.DataFrame(iter_rows)
iter_timing["gap+overlay_s"] = iter_timing["gap_s"] + iter_timing["overlay_s"]
print(
    f"sum gap_s={iter_timing['gap_s'].sum():.3f}s  "
    f"sum overlay_s={iter_timing['overlay_s'].sum():.3f}s  "
    f"iters={len(iter_timing)}"
)
iter_timing

sum gap_s=19.902s  sum overlay_s=0.038s  iters=15


,iter,gap_s,overlay_s,interest_delta_max,stopped,gap+overlay_s
0,1,1.286,0.002,546.719,,1.288
1,2,1.316,0.002,228.262,,1.318
2,3,1.226,0.002,100.737,,1.229
3,4,1.255,0.002,45.377,,1.257
4,5,1.216,0.002,20.414,,1.219
5,6,1.252,0.002,6.323,,1.254
6,7,1.172,0.003,1.287,,1.174
7,8,1.233,0.002,0.189,,1.235
8,9,1.144,0.003,0.021,,1.147
9,10,1.242,0.003,0.002,,1.244


## 6. `external_interest()` cost during one gap call

Wraps `MacroDebtBook.external_interest` to count calls and cumulative time while
running a single `external_residual_borrowing` (the hot function inside each gap iter).

In [7]:
interest_stats = {"calls": 0, "seconds": 0.0}
_orig_ext_int = MacroDebtBook.external_interest


def _traced_external_interest(self):
    t0 = time.perf_counter()
    out = _orig_ext_int(self)
    interest_stats["seconds"] += time.perf_counter() - t0
    interest_stats["calls"] += 1
    return out


MacroDebtBook.external_interest = _traced_external_interest  # type: ignore[method-assign]
try:
    t0 = time.perf_counter()
    _ = external_residual_borrowing(
        path.baseline,
        path.shocked,
        resfin_interest=None,
        **dyn._borrow_kwargs(),  # type: ignore[arg-type]
    )
    gap_call_s = time.perf_counter() - t0
finally:
    MacroDebtBook.external_interest = _orig_ext_int  # type: ignore[method-assign]

interest_probe = pd.Series(
    {
        "external_residual_borrowing_s": gap_call_s,
        "external_interest_calls": interest_stats["calls"],
        "external_interest_s": interest_stats["seconds"],
        "external_interest_share_%": (
            100.0 * interest_stats["seconds"] / gap_call_s if gap_call_s else 0.0
        ),
        "seconds_per_call": (
            interest_stats["seconds"] / interest_stats["calls"]
            if interest_stats["calls"]
            else 0.0
        ),
    }
)
interest_probe

external_residual_borrowing_s    1.353
external_interest_calls         54.000
external_interest_s              1.308
external_interest_share_%       96.676
seconds_per_call                 0.024
dtype: float64

## 7. Suite batch vs sum of individuals

`StressSuite.run_external_standard` (no B2) for comparison with section 2.

In [8]:
suite_rows: list[dict] = []
with timed("StressSuite.run_external_standard", suite_rows):
    suite_out = StressSuite(context=ctx).run_external_standard()
with timed("B2_PrimaryBalance (coupled)", suite_rows):
    _ = CoupledScenarioRunner(context=ctx).run(
        ScenarioRegistry.get("B2_PrimaryBalance")
    )

suite_timing = timing_frame(suite_rows)
print("suite keys:", list(suite_out))
suite_timing

suite keys: ['A1_Historical', 'B1_GDP', 'B3_Exports', 'B4_OtherFlows', 'B5_FX', 'B6_Combo']


,section,seconds,share_%
0,StressSuite.run_external_standard,95.240,97.293
1,B2_PrimaryBalance (coupled),2.650,2.707


## 8. Summary

In [9]:
summary = pd.concat(
    [
        load_timing.assign(group="load"),
        scenario_timing.rename(columns={"scenario": "section"}).assign(
            group="standard_external"
        )[["group", "section", "seconds", "share_%"]],
        tailored_timing.rename(columns={"scenario": "section"}).assign(
            group="tailored_external"
        )[["group", "section", "seconds", "share_%"]],
    ],
    ignore_index=True,
)

by_group = (
    summary.groupby("group", sort=False)["seconds"]
    .sum()
    .rename("seconds")
    .to_frame()
)
by_group["share_%"] = 100.0 * by_group["seconds"] / by_group["seconds"].sum()

print("Totals by group")
display(by_group)
print("\nB3 pipeline stages")
display(stage_timing)
print("\nexternal_interest probe (one gap call)")
display(interest_probe.to_frame("value"))
summary.sort_values("seconds", ascending=False).head(20)

Totals by group


,seconds,share_%
group,,
load,30.335,20.616
standard_external,93.994,63.879
tailored_external,22.814,15.505



B3 pipeline stages


,section,seconds,share_%
0,1_shock_apply,0.007,0.036
1,2_ExternalDebtDynamics.from_context,0.000,0.000
2,3_compute_gap_converged,19.069,99.954
3,4_build_external_overlay_final,0.002,0.010
4,5_StressExternalRatios.from_path,0.000,0.000



external_interest probe (one gap call)


,value
external_residual_borrowing_s,1.353
external_interest_calls,54.000
external_interest_s,1.308
external_interest_share_%,96.676
seconds_per_call,0.024


,section,seconds,share_%,group
0,load_core,29.776,98.157,load
14,C3_Commodity,18.847,82.612,tailored_external
5,A1_Historical,18.536,19.721,standard_external
10,B6_Combo,17.997,19.147,standard_external
7,B3_Exports,17.909,19.054,standard_external
8,B4_OtherFlows,17.885,19.028,standard_external
9,B5_FX,17.718,18.850,standard_external
13,C1_CombinedCL,2.655,11.640,tailored_external
11,B2_PrimaryBalance,2.589,2.754,standard_external
6,B1_GDP,1.359,1.446,standard_external
